In [3]:
# ライブラリーのインポート
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# データの読み込み
data_trained = pd.read_csv('/kaggle/input/titanic/train.csv')
data_test = pd.read_csv('/kaggle/input/titanic/test.csv')
data_gender_submission = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

## 特徴量エンジニアリング
文字列を数値に置き換え
欠損値の扱い

In [18]:
# 教師用データとテストデータ、双方にエンジニアリングを行うため、一旦一つに結合する。
    # データの中身を変更していく際にテストデータには、教師データに実行した変更が反映されてない状態になるから

# 結合は concatでリスト指定。教師、テストで縦連結。
data_all = pd.concat([data_trained, data_test], sort = False) # 列の順番は変えずにただ結合してください

data_all


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,NaN,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,NaN,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [19]:
# 各特徴量の欠損値の数を確認
# データの欠損は　isnull
# TestデータはSurvivedがないので欠損値扱い

data_all.isnull().sum()

# Null418件がテストデータの数を意味する

PassengerId       0
Survived        418
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
dtype: int64

In [20]:
# Sexを数値に置き換え
data_all['Sex'].replace(['male', 'female'], [0, 1], inplace = True) # 変更後の結果をでーたふれ

/tmp/ipykernel_17/851876621.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_all['Sex'].replace(['male', 'female'], [0, 1], inplace = True) # 変更後の結果をでーたふれ
/tmp/ipykernel_17/851876621.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_all['Sex'].replace(['male', 'female'], [0, 1], inpla

In [21]:
data_all.head()
# Sexの特徴量が数値化された

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S


In [22]:
# Embarkedの欠損値を補完する、数値に置き換える
    # 2件でも欠損があればアルゴリズムは適応できないから
    # 本来なら欠損値を補完する方法を検討するが、今回は2件だし一番多いSで補完
data_all['Embarked'].fillna('S', inplace = True)
data_all['Embarked'].replace(['S', 'C', 'Q'], [0, 1, 2], inplace = True) # 変更後の結果をでーたふれ
data_all.head(900)

# One-Hotエンコーディング
    # Embarkedを1、2、3で置き換えると、その大きさに意味があるものとアルゴリズムに勘違いされてしまう場合がある。ので、データの数分、列数が増えるので、データサイズが大きくなる。
    # カテゴリ指定を予めできるものもあり、数値特有の大きさや順番を考慮しないものもある

/tmp/ipykernel_17/2239516266.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_all['Embarked'].fillna('S', inplace = True)
/tmp/ipykernel_17/2239516266.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_all['Embarked'].replace(['S', 'C', 'Q'], [0, 1, 2], inplace = True) # 変更後の結果をでーたふれ


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,0
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,1
2,3,1.0,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,0
4,5,0.0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4,896,NaN,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1,22.0,1,1,3101298,12.2875,NaN,0
5,897,NaN,3,"Svensson, Mr. Johan Cervin",0,14.0,0,0,7538,9.2250,NaN,0
6,898,NaN,3,"Connolly, Miss. Kate",1,30.0,0,0,330972,7.6292,NaN,2
7,899,NaN,2,"Caldwell, Mr. Albert Francis",0,26.0,1,1,248738,29.0000,NaN,0


In [23]:
data_all.isnull().sum()

PassengerId       0
Survived        418
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          0
dtype: int64

In [24]:
# Fareは平均で補完, Ageも平均で補完（Ageはそもそも欠損が多いので補完するかどうかを検討した方がいい）
data_all['Fare'].fillna(np.mean(data_all['Fare']), inplace = True)
data_all['Age'].fillna(np.mean(data_all['Age']), inplace = True)
data_all.head(900)

/tmp/ipykernel_17/828574674.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_all['Fare'].fillna(np.mean(data_all['Fare']), inplace = True)
/tmp/ipykernel_17/828574674.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,0
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,1
2,3,1.0,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,0
4,5,0.0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4,896,NaN,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1,22.0,1,1,3101298,12.2875,NaN,0
5,897,NaN,3,"Svensson, Mr. Johan Cervin",0,14.0,0,0,7538,9.2250,NaN,0
6,898,NaN,3,"Connolly, Miss. Kate",1,30.0,0,0,330972,7.6292,NaN,2
7,899,NaN,2,"Caldwell, Mr. Albert Francis",0,26.0,1,1,248738,29.0000,NaN,0


In [25]:
# 影響のないあるいは少なそうな特徴量については一旦削除。
# drop 行または列で削除する関数。axis - 1で列方向
drop_colums = ['PassengerId', 'Name', 'Parch', 'SibSp', 'Ticket', 'Cabin']
data_all.drop(drop_colums, axis = 1, inplace = True)
data_all.isnull().sum()
# まあこれでひとまず欠損値はなくなった
# SibSpやらParchは消していいのか？

Survived    418
Pclass        0
Sex           0
Age           0
Fare          0
Embarked      0
dtype: int64

In [26]:
# 結合したデータを、再度、教師データと、テストデータに分ける
# len関数は、行数を返してくれる。スライスで該当行を抜き出し。
# 先頭行から教師データの件数だけ抜き出すのと、その一個下のデータから
data_trained = data_all[:len(data_trained)]
data_test = data_all[len(data_trained):]

## モデル作成


In [27]:
# アルゴリズムに投入。多くの場合別々に投入するのでそのために特徴量と目的変数を分離、x,yと頭文字をつけておく
y_data_trained = data_trained['Survived']
x_data_trained = data_trained.drop('Survived', axis = 1) # 説明変数を格納、dropで目的変数は削除。axis = 1は列を削除の意味
x_data_test = data_test.drop('Survived', axis = 1) # testデータにはSurvivedがなかったが、結合で列が発生したので消しておく。

In [28]:
# LogisticRegression アルゴリズムをインポート
    # 説明変数の対数をとって、線形回帰を確率0-1に変換したものでニ値分類に適用される
    # 多くのライブラリがあるが、とりまロジスティック回帰を行なって今後の指針にすることが多い。多値分類だと決定木とかランダムフォレストを使う。解釈もシンプルなので。
    # アルゴリズムチートシートなるものがあるから、それを使って何を使えばいいか
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(penalty = 'l2', solver = 'sag', random_state = 0) 
# 生存率、パラメータの探索方法、乱数のシード（指定で実行結果が毎回同じになる。0に意味はない。同じ数字を振ったものが同じ結果になる）

# 教師データに適用して、学習させる、変更なし。
clf.fit(x_data_trained, y_data_trained) # 説明変数と目的変数を別の変数として指定

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(random_state=0, solver='sag')

In [29]:
y_data_pred = clf.predict(x_data_test)# 学習したclfに対して、引数にはテストデータを指定
# predictの閾値はデフォルト0.5。それ未満を0、以上を1として丸めて返す。本来0〜1の確率として返す。生存確率が高い0.5以上を生存、未満を0の死亡と返す。
y_data_pred
# Survivedに対する予測結果。テストデータとして与えた先頭行の乗船客から順に予測が行われリストが返される。

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
       0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0.,
       0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1.,
       0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0., 0.,
       0., 0., 0., 0., 0.

## Submit

In [30]:
# 予測結果をSubmit用のCSVファイルとして作成
submit = data_gender_submission # 冒頭で作った変数。提出用CSVのこと。めんどいのでsubmitとする。
submit['Survived'] = list(map(int, y_data_pred)) 
# map 指定した要素に対して、適用したい関数（int）を適用できる関数。予測結果を整数型にして提出フォーマットに合わせるため。
# リストで提出用ファイルに.
submit.to_csv('logisticregression_submit.csv', index = False) # 出力ファイル名,index番号は出力しない